# exp163 — multi-draft contact prediction, interactively

A contacts-v1 1.5B fine-tuned to emit **many candidate contact maps in one generation**.
Given a protein sequence it writes a chain of `<begin_statements>` sections, each an
independent guess at the contact map; the best is substantially better than any
single-shot prediction.

Measured on 553 held-out proteins (4 rollouts each):

| | value |
|---|---|
| sections per generation | ~15 |
| Jaccard between sections | 0.071 (nearly disjoint) |
| first / last / **best** section F1 | 0.184 / 0.249 / **0.303** |
| base model, single shot | 0.237 |

**Best-of-N is +27% over the base model** (paired, +26σ). That spread is the point —
it is what a best-of-N RL reward selects over.

Weights come from the public `open-athena/MarinFold` bucket; no token needed.

**Runtime → Change runtime type → GPU** (T4 is enough; 1.5B in bf16).

In [ ]:
%pip install -q "huggingface_hub>=1.5" "transformers>=4.53" torch matplotlib numpy

In [ ]:
# --- fetch the model (anonymous; ~2.9 GB) --------------------------------------
from huggingface_hub import HfApi
from pathlib import Path

BUCKET = "open-athena/MarinFold"
PREFIX = "checkpoints/plm-exp163-refine-cv1-1_5b-lr1e-4-e1-cos-tpuF/hf/step-404"
LOCAL = Path("/content/exp163_model"); LOCAL.mkdir(parents=True, exist_ok=True)

api = HfApi()
paths = [p.path for p in api.list_bucket_tree(BUCKET) if p.path.startswith(PREFIX)]
print(f"{len(paths)} files")
api.download_bucket_files(BUCKET, paths, local_dir=str(LOCAL), token=False)
MODEL_DIR = LOCAL / PREFIX
print(sorted(p.name for p in MODEL_DIR.iterdir()))

In [ ]:
import torch, numpy as np, re
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR, torch_dtype=torch.bfloat16, device_map="auto").eval()

# The rope config must resolve. If both are None the checkpoint carries the newer
# `rope_parameters`-only spelling, which transformers <5 silently ignores -> default
# rope -> predictions that degrade with sequence length. That bug cost this project a
# full round of invalidated results, so it is asserted rather than assumed.
assert getattr(model.config, "rope_theta", None) or getattr(model.config, "rope_scaling", None), \
    "rope config did not load — see stage_v3_to_gcs.py"
print("rope_theta:", getattr(model.config, "rope_theta", None))
print("params: %.2fB" % (sum(p.numel() for p in model.parameters()) / 1e9))

In [ ]:
# --- the contacts-v1 document format -------------------------------------------
# <contacts-v1.multi> <begin_sequence> (position, residue) pairs in RANDOM order, then
# a chain of <begin_statements> sections of `<contact> <pi> <pj>` triples. The shuffle
# is deliberate: the model must use the <pN> position tokens, not prompt adjacency.

AA3 = {"A": "ALA", "R": "ARG", "N": "ASN", "D": "ASP", "C": "CYS", "Q": "GLN",
       "E": "GLU", "G": "GLY", "H": "HIS", "I": "ILE", "L": "LEU", "K": "LYS",
       "M": "MET", "F": "PHE", "P": "PRO", "S": "SER", "T": "THR", "W": "TRP",
       "Y": "TYR", "V": "VAL"}
P0 = 194           # first position token used by the corpus builder
MIN_SEP = 6        # pairs closer than this are trivially predictable; excluded

def build_prompt(seq, seed=0):
    """contacts-v1.multi header for `seq`. Returns (prompt, seq_positions)."""
    seq = seq.strip().upper()
    L = len(seq)
    pos = [P0 + i for i in range(L)]
    order = np.random.default_rng(seed).permutation(L)
    toks = ["<contacts-v1.multi>", "<begin_sequence>"]
    for i in order:
        if i == 0:
            toks.append("<n-term>")
        if i == L - 1:
            toks.append("<c-term>")
        toks += [f"<p{pos[i]}>", "<" + AA3.get(seq[i], "ALA") + ">"]
    toks.append("<begin_statements>")
    return " ".join(toks), pos

CONTACT = re.compile(r"<contact> <p(\d+)> <p(\d+)>")

def parse_sections(text, pos):
    """One contact set per section. The prompt already supplied the FIRST marker, so
    chunk 0 holds section 1 and must not be dropped."""
    idx = {p: i for i, p in enumerate(pos)}
    chunks = text.split("<begin_statements>")
    keep = chunks if CONTACT.search(chunks[0]) else chunks[1:]
    out = []
    for c in keep:
        s = {tuple(sorted((idx[int(a)], idx[int(b)])))
             for a, b in CONTACT.findall(c) if int(a) in idx and int(b) in idx}
        out.append({(i, j) for i, j in s if abs(i - j) >= MIN_SEP})
    return [s for s in out if s]

In [ ]:
@torch.no_grad()
def predict(seq, max_sections=8, temperature=1.0, top_p=0.95, seed=0):
    """Generate a chain of candidate contact maps for `seq`.

    `max_sections` is a REAL cap. The model does not reliably emit <end> (only ~56%
    of generations terminate on their own), so the token budget is sized from the
    measured section length AND the parsed output is truncated."""
    prompt, pos = build_prompt(seq, seed=seed)
    ids = tok(prompt, add_special_tokens=False, return_tensors="pt").to(model.device)
    per_section = 3 * 220 + 8            # ~220 contacts/section, measured
    budget = min(8192 - ids.input_ids.shape[1] - 8, per_section * max_sections)
    torch.manual_seed(seed)
    out = model.generate(**ids, do_sample=True, temperature=temperature, top_p=top_p,
                         max_new_tokens=budget,
                         eos_token_id=tok.convert_tokens_to_ids("<end>"),
                         pad_token_id=tok.convert_tokens_to_ids("<pad>"))
    text = tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=False)
    return parse_sections(text, pos)[:max_sections], len(seq)

SEQ = ("MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLSGAEKAVQVKVKALPDAQFEVVHSLAKWKR"
       "QTLGQHDFSAGEGLYTHMKALRPDEDRLSPLHSVYVDQWDWERVMGDGERQFSTLKSTVEAIWAGIKATEAAVSEEF")
sections, L = predict(SEQ, max_sections=8, seed=0)
print(f"L={L}, {len(sections)} sections, sizes: {[len(s) for s in sections]}")

In [ ]:
# --- visualise every candidate contact map --------------------------------------
import matplotlib.pyplot as plt

def plot_sections(sections, L, ncol=4, gt=None):
    n = len(sections)
    nrow = (n + ncol - 1) // ncol
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.1 * ncol, 3.1 * nrow), squeeze=False)
    for k, ax in enumerate(axes.ravel()):
        if k >= n:
            ax.axis("off")
            continue
        M = np.zeros((L, L))
        for i, j in sections[k]:
            M[i, j] = M[j, i] = 1
        title = f"section {k + 1}  ({len(sections[k])})"
        if gt is not None:
            # upper triangle = prediction, lower = truth, for direct comparison
            G = np.zeros((L, L))
            for i, j in gt:
                G[i, j] = G[j, i] = 1
            M = np.triu(M) + np.tril(G) * 0.55
            tp = len(sections[k] & gt)
            f1 = 2 * tp / (len(sections[k]) + len(gt)) if (sections[k] or gt) else 0
            title += f"  F1={f1:.3f}"
        ax.imshow(M, cmap="Greys", interpolation="nearest", origin="lower")
        ax.set_title(title, fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle("candidate contact maps — one per generated section"
                 + ("  (upper = predicted, lower = true)" if gt is not None else ""),
                 fontsize=11)
    fig.tight_layout()
    plt.show()

plot_sections(sections, L)

In [ ]:
# --- how much do the candidates actually differ? --------------------------------
# This is the property best-of-N depends on. Near-copies give a flat reward no matter
# how accurate the model is.
J = np.eye(len(sections))
for i, a in enumerate(sections):
    for j, b in enumerate(sections):
        J[i, j] = len(a & b) / max(1, len(a | b))
off = J[~np.eye(len(sections), dtype=bool)]
print(f"pairwise Jaccard: mean {off.mean():.3f}  max {off.max():.3f}")
print(f"union of all sections: {len(set().union(*sections))} contacts "
      f"vs largest single section {max(len(s) for s in sections)}")
plt.figure(figsize=(4.4, 3.6))
plt.imshow(J, cmap="viridis", vmin=0, vmax=1)
plt.colorbar(label="Jaccard")
plt.title("section-vs-section overlap")
plt.xlabel("section"); plt.ylabel("section")
plt.tight_layout(); plt.show()

In [ ]:
# --- knobs worth turning --------------------------------------------------------
# temperature controls the spread: lower -> sections converge (less for best-of-N to
# select over); higher -> more diverse but individually worse.
for T in (0.7, 1.0, 1.3):
    secs, _ = predict(SEQ, max_sections=6, temperature=T, seed=1)
    if len(secs) < 2:
        print(f"T={T}: {len(secs)} section(s)")
        continue
    js = [len(a & b) / max(1, len(a | b))
          for i, a in enumerate(secs) for b in secs[i + 1:]]
    print(f"T={T}: {len(secs)} sections, sizes {[len(s) for s in secs]}, "
          f"mean Jaccard {np.mean(js):.3f}")

## Scoring against a known structure

With ground-truth contacts (0-based sequence-index pairs, separation ≥ 6) you can see
per-section F1 and the best-of-N gain — the quantity an RL reward would maximise:

```python
gt = {(3, 40), (5, 42), ...}
plot_sections(sections, L, gt=gt)
f1 = [2 * len(s & gt) / (len(s) + len(gt)) for s in sections]
print(f"first {f1[0]:.3f} | last {f1[-1]:.3f} | BEST {max(f1):.3f}")
```

### Things worth knowing

* **Successive sections do not reliably improve** (`frac_improving` ≈ 0.55, barely
  above chance). The value is in the *spread*, not a refinement trajectory — reward the
  best section, not the last.
* **Conditioning on someone else's drafts hurts.** Pasting external rollouts into the
  prompt degrades prediction monotonically (−0.035 F1 at 4 drafts, −0.117 at 16).
  Self-generated chains help; externally supplied ones do not.
* **The model often does not stop** — only ~56% of generations emit `<end>`. Always cap
  `max_sections`.
* `MIN_SEP = 6`: pairs closer than 6 apart in sequence are excluded as trivially
  predictable from the backbone.